# Week 46, Day 1: Holding a location in mind: persistent activity and bump attractors

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w046/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Error variance grows with delay

Complete run(): on each step update every trial's activity u by dt (-u + W f(u) + input) plus noise of size noise times sqrt(dt); record the bump centre of every trial every 500 delay steps. Then compute the variance of the remembered location at each recorded delay.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(2000)
N = 60; th = np.linspace(-np.pi, np.pi, N, endpoint=False); D = th[:, None] - th[None, :]
f = lambda x: np.clip(x, 0, 1)

def ring(J1=6.0, J0=-2.0):
    return (J0 + J1 * np.cos(D)) / N

def run(W, cue=0.0, delay=3000, noise=0.05, trials=80, dist=None, amp=0.3, dt=0.1):
    U = np.zeros((trials, N)); cen = []
    for k in range(300 + delay):
        inp = 0.1 + (0.5 * np.cos(th - cue) if k < 300 else 0)
        if dist is not None and 300 + dist[1] <= k < 400 + dist[1]:
            inp = inp + amp * np.exp(5 * (np.cos(th - dist[0]) - 1))
        ...
        if k >= 300 and (k - 300) % 500 == 499:
            cen.append(np.angle((f(U) * np.exp(1j * th)).sum(1)))
    return np.array(cen), f(U)

cen, _ = run(ring())
var = np.var(np.angle(np.exp(1j * cen)), 1)
delays = 50 * np.arange(1, len(var) + 1)       # in units of tau: 500 steps of 0.1
print("variance of remembered location by delay:", np.round(var, 4))
plt.plot(delays, var, "o-"); plt.xlabel("delay (units of tau)"); plt.ylabel("variance (rad^2)"); plt.show()

In [ ]:
import numpy as np
def test_linear():
    "the variance of the remembered location grows roughly linearly with delay"
    ratio = var[-1] / var[0]
    assert 4 < ratio < 9
def test_mean():
    "the memory is unbiased on average"
    assert abs(np.angle(np.exp(1j * cen[-1])).mean()) < 0.05

test_linear()
test_mean()
print("All checks passed")

### Exercise: Near and far distractors

For distractors at 0.3, 0.6, 1.2 and 2.5 rad, presented 1,000 steps into the delay with amplitude 0.3, run 60 trials with noise 0.02 and record the median final location.

In [ ]:
shift = {}
for d in [0.3, 0.6, 1.2, 2.5]:
    c, _ = ...
    shift[d] = np.median(np.angle(np.exp(1j * c[-1])))
    print(f"distractor at {d:.1f} rad: memory ends at {shift[d]:+.2f} rad")

In [ ]:
import numpy as np
def test_near():
    "a distractor 0.3 rad away pulls the memory more than 0.15 rad towards it"
    assert shift[0.3] > 0.15
def test_far():
    "a distractor 2.5 rad away leaves the memory within 0.15 rad of the cue"
    assert abs(shift[2.5]) < 0.15

test_near()
test_far()
print("All checks passed")

### Exercise: Below the critical strength

Run 20 noiseless trials with J1 = 6 and with J1 = 2.5 and record the mean peak activity at the end of the delay.

In [ ]:
peak = {}
for J1 in [6.0, 2.5]:
    _, R = ...
    peak[J1] = R.max(1).mean()
    print(f"J1 = {J1}: peak activity at the end of the delay {peak[J1]:.3f}")

In [ ]:
import numpy as np
def test_hold():
    "with J1 = 6 the bump persists through the delay"
    assert peak[6.0] > 0.4
def test_fade():
    "with J1 = 2.5 the activity has decayed to near background"
    assert peak[2.5] < 0.15

test_hold()
test_fade()
print("All checks passed")